# Team Season - team_dash_pt_pass

This notebook inspects the data **downloaded** from the `team_dash_pt_pass` endpoint.

Source folder: `02_data/01_raw/2025_26/02_team_season/team_dash_pt_pass`

Scope:
- Read parquet files in the folder
- Show how many files were downloaded and how many rows/columns they contain
- Report missing values (null cells) overall and by row/column distribution

Notes:
- Read-only analysis: no exports or writes


In [17]:
from pathlib import Path
import pandas as pd
import numpy as np

# Resolve project root by walking up to find 02_data
project_root = Path.cwd()
for _ in range(6):
    if (project_root / "02_data").exists():
        break
    project_root = project_root.parent

endpoint = "team_dash_pt_pass"
data_dir = project_root / "02_data" / "01_raw" / "2025_26" / "02_team_season" / endpoint

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)


In [18]:
files = sorted(data_dir.glob("*.parquet"))
print(f"Endpoint: {endpoint}")
print(f"Folder: {data_dir}")
print(f"Parquet files: {len(files)}")

files_df = pd.DataFrame({
    "file": [f.name for f in files],
    "size_mb": [round(f.stat().st_size / 1e6, 3) for f in files],
})
files_df


Endpoint: team_dash_pt_pass
Folder: /Users/pablodiazgonzalez/Documents/MachineLearning/AlgoBasket/02_data/01_raw/2025_26/02_team_season/team_dash_pt_pass
Parquet files: 30


,file,size_mb
0,team_dash_pt_pass__team_id=1610612737.parquet,0.017
1,team_dash_pt_pass__team_id=1610612738.parquet,0.016
2,team_dash_pt_pass__team_id=1610612739.parquet,0.017
3,team_dash_pt_pass__team_id=1610612740.parquet,0.017
4,team_dash_pt_pass__team_id=1610612741.parquet,0.018
5,team_dash_pt_pass__team_id=1610612742.parquet,0.017
6,team_dash_pt_pass__team_id=1610612743.parquet,0.016
7,team_dash_pt_pass__team_id=1610612744.parquet,0.017
8,team_dash_pt_pass__team_id=1610612745.parquet,0.016
9,team_dash_pt_pass__team_id=1610612746.parquet,0.017


In [19]:
dfs = [pd.read_parquet(f) for f in files]

per_file = pd.DataFrame({
    "file": [f.name for f in files],
    "rows": [len(df) for df in dfs],
    "cols": [df.shape[1] for df in dfs],
})
per_file


,file,rows,cols
0,team_dash_pt_pass__team_id=1610612737.parquet,46,23
1,team_dash_pt_pass__team_id=1610612738.parquet,38,23
2,team_dash_pt_pass__team_id=1610612739.parquet,42,23
3,team_dash_pt_pass__team_id=1610612740.parquet,38,23
4,team_dash_pt_pass__team_id=1610612741.parquet,56,23
5,team_dash_pt_pass__team_id=1610612742.parquet,43,23
6,team_dash_pt_pass__team_id=1610612743.parquet,34,23
7,team_dash_pt_pass__team_id=1610612744.parquet,38,23
8,team_dash_pt_pass__team_id=1610612745.parquet,30,23
9,team_dash_pt_pass__team_id=1610612746.parquet,47,23


In [20]:
df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

print(f"Combined shape: {df.shape}")
print(f"Total rows: {len(df)}")
print(f"Total columns: {df.shape[1] if len(df.columns) else 0}")


Combined shape: (1221, 23)
Total rows: 1221
Total columns: 23


In [21]:
rows, cols = df.shape
total_cells = rows * cols
null_cells = int(df.isna().sum().sum()) if total_cells else 0
null_pct = (null_cells / total_cells) if total_cells else 0

summary = pd.DataFrame({
    "rows": [rows],
    "cols": [cols],
    "total_cells": [total_cells],
    "null_cells": [null_cells],
    "null_pct": [round(null_pct * 100, 3)],
})
summary


,rows,cols,total_cells,null_cells,null_pct
0,1221,23,28083,1221,4.348


In [22]:
if df.empty:
    col_summary = pd.DataFrame()
else:
    col_nulls = df.isna().sum()
    col_null_pct = df.isna().mean()
    col_summary = (
        pd.DataFrame({
            "null_cells": col_nulls,
            "null_pct": (col_null_pct * 100).round(3),
        })
        .sort_values("null_pct", ascending=False)
    )

col_summary


,null_cells,null_pct
PASS_TO,611,50.041
PASS_FROM,610,49.959
TEAM_ID,0,0.000
FG2M,0,0.000
SEASON_TYPE,0,0.000
SEASON,0,0.000
TABLE_NAME,0,0.000
FG3_PCT,0,0.000
FG3A,0,0.000
FG3M,0,0.000


In [23]:
if df.empty:
    row_dist = pd.DataFrame()
else:
    row_null_pct = df.isna().mean(axis=1)
    bins = [0, 0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 1.0]
    row_bins = pd.cut(row_null_pct, bins=bins, include_lowest=True)
    row_counts = row_bins.value_counts().sort_index()
    row_dist = pd.DataFrame({
        "rows": row_counts,
        "pct_rows": (row_counts / len(row_null_pct) * 100).round(3),
    })

row_dist


,rows,pct_rows
"(-0.001, 0.01]",0,0.0
"(0.01, 0.05]",1221,100.0
"(0.05, 0.1]",0,0.0
"(0.1, 0.25]",0,0.0
"(0.25, 0.5]",0,0.0
"(0.5, 0.75]",0,0.0
"(0.75, 1.0]",0,0.0


In [24]:
# Merge all teams and export

files = sorted(data_dir.glob("*.parquet"))
dfs = [pd.read_parquet(f) for f in files]
df_all = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

if df_all.empty:
    print("No data to merge.")
else:
    out_dir = project_root / "02_data" / "02_cleaned" / "2025_26" / "02_team_season"
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / "team_dash_pt_pass.parquet"
    df_all.to_parquet(out_path, index=False)
    print(f"Saved to: {out_path}")


Saved to: /Users/pablodiazgonzalez/Documents/MachineLearning/AlgoBasket/02_data/02_cleaned/2025_26/02_team_season/team_dash_pt_pass.parquet
